# FT03 — Data Discovery: USGS

## Objetivo
Explorar a API pública USGS Water Data API para identificar
endpoints, parâmetros, estruturas JSON e possibilidades
de utilização no pipeline ETL.

## Ambiente
- Python 3.13.0
- Pandas 3.1.1
- Requests 2.34.2
- Jupyter Notebook

## Etapas
1. Identificação dos endpoints.
2. Primeira requisição HTTP.
3. Análise do status HTTP.
4. Inspeção da resposta JSON.
5. Identificação de estruturas aninhadas.
6. Avaliação dos dados para transformação.

## 1. Importação de bibliotecas e validação do ambiente

### Objetivo
Importar as bibliotecas Python necessárias para a investigação da API USGS e verificar as versões do interpretador e das principais dependências utilizadas no projeto.

### Justificativa
Antes de iniciar a extração e manipulação dos dados, precisamos garantir que o ambiente de execução esteja corretamente configurado.

A validação permite identificar:
- Qual versão do Python está sendo utilizada.
- Qual interpretador está executando o notebook.
- Quais versões das bibliotecas estão instaladas.
- Se as dependências necessárias estão disponíveis.

### Bibliotecas utilizadas

| Biblioteca | Finalidade |
|---|---|
| `sys` | Acessar informações do ambiente Python e do interpretador. |
| `pandas` | Manipulação, transformação e análise de dados tabulares. |
| `requests` | Realizar requisições HTTP para consumir APIs. |
| `json` | Interpretar e manipular estruturas de dados JSON. |
| `re` | Trabalhar com expressões regulares para pesquisa textual. |

### Explicação técnica

- `import`: carrega uma biblioteca ou módulo para utilização no código.
- `as pd`: cria um apelido para a biblioteca Pandas, seguindo a convenção amplamente utilizada na comunidade Python.
- `sys.version`: retorna informações sobre a versão do Python.
- `sys.executable`: informa o caminho do executável Python utilizado pelo kernel.
- `pd.__version__`: retorna a versão instalada do Pandas.
- `requests.__version__`: retorna a versão instalada do Requests.
- `print()`: apresenta as informações no console de saída do notebook.

### Resultado da execução

- Python: 3.13.0
- Pandas: 3.1.1
- Requests: 2.34.2
- Interpretador: ambiente virtual `.venv` do projeto FT03-Python.

### Conclusão

O ambiente de execução foi identificado e as bibliotecas principais foram carregadas com sucesso. O notebook está preparado para iniciar as requisições à API USGS e realizar a exploração dos dados.

In [14]:
import sys
import pandas as pd
import requests
import json
import re

print("Python:", sys.version)
print("Interpretador:", sys.executable)
print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)

Python: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
Interpretador: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\.venv\Scripts\python.exe
Pandas: 3.1.1
Requests: 2.34.2


## 2. Configuração e primeira requisição à API USGS

### Objetivo

Configurar o acesso à API pública USGS Water Data API e realizar a primeira requisição HTTP para consultar o catálogo de características ambientais disponíveis.

### Justificativa

Precisamos estabelecer uma comunicação entre nosso ambiente Python e a fonte externa de dados.

Essa etapa permite verificar:
- Se o endpoint da API está acessível.
- Se a requisição HTTP foi processada com sucesso.
- Qual URL foi efetivamente consultada.
- Qual formato de conteúdo foi retornado pelo serviço.

### Conceitos técnicos

**API (Application Programming Interface):** interface que permite a comunicação entre diferentes sistemas por meio de regras e estruturas previamente definidas.

**HTTP (Hypertext Transfer Protocol):** protocolo utilizado para comunicação entre clientes e servidores na web.

**Endpoint:** endereço específico de uma API responsável por disponibilizar determinado recurso ou funcionalidade.

**GET:** método HTTP utilizado para solicitar informações de um servidor.

**Parâmetros de consulta:** informações adicionais enviadas na URL para definir características da solicitação.

### Configurações utilizadas

| Variável | Finalidade |
|---|---|
| `BASE_URL` | Define o endereço-base da API USGS. |
| `ENDPOINT` | Define o recurso específico que será consultado. |
| `PARAMS` | Armazena os parâmetros enviados na requisição. |
| `response` | Armazena a resposta HTTP recebida do servidor. |

### Explicação dos principais comandos

- `f-string`: permite inserir variáveis diretamente em uma string.
- `requests.get()`: realiza uma requisição HTTP utilizando o método GET.
- `params=PARAMS`: envia os parâmetros de consulta à API.
- `timeout=30`: estabelece um limite de 30 segundos para a espera pela resposta.
- `response.status_code`: retorna o código HTTP recebido.
- `response.url`: apresenta a URL efetivamente consultada.
- `response.headers.get("Content-Type")`: identifica o tipo de conteúdo informado pelo servidor.


In [15]:
# Configuração da API USGS

BASE_URL = "https://api.waterdata.usgs.gov/samples-data"

ENDPOINT = f"{BASE_URL}/codeservice/characteristics"

PARAMS = {
    "mimeType": "application/json"
}

# Requisição HTTP
response = requests.get(
    ENDPOINT,
    params=PARAMS,
    timeout=30
)

# Resultado da conexão
print("Status HTTP:", response.status_code)
print("URL consultada:", response.url)
print("Content-Type:", response.headers.get("Content-Type"))

Status HTTP: 200
URL consultada: https://api.waterdata.usgs.gov/samples-data/codeservice/characteristics?mimeType=application%2Fjson
Content-Type: application/json


### Resultado da execução

| Indicador | Resultado |
|---|---|
| Status HTTP | 200 |
| Content-Type | application/json |
| Endpoint | codeservice/characteristics |
| Tempo de execução observado | 2,8 segundos |

### Interpretação

O código HTTP 200 indica que a requisição foi processada com sucesso pelo servidor.

O cabeçalho `Content-Type: application/json` informa que a resposta está sendo disponibilizada no formato JSON.

A URL apresentada confirma o envio do parâmetro `mimeType=application/json`.

### Conclusão

A comunicação entre o ambiente Python e a API USGS foi estabelecida com sucesso.

A resposta recebida confirma que podemos avançar para a inspeção do conteúdo JSON e investigar a estrutura dos dados disponibilizados pela API.

## 3. Conversão e inspeção da resposta JSON

### Objetivo

Converter a resposta HTTP recebida da API USGS em um objeto Python, identificar sua estrutura hierárquica e realizar uma inspeção inicial dos registros retornados.

### Justificativa

A resposta de uma API geralmente é disponibilizada em formatos estruturados, como JSON.

Embora esse formato seja adequado para comunicação entre sistemas, precisamos convertê-lo em estruturas Python para acessar seus elementos, investigar os campos disponíveis e preparar os dados para posterior transformação em DataFrame.

### Conceitos técnicos

**JSON (JavaScript Object Notation):** formato textual utilizado para representar dados estruturados, organizados em objetos, listas, pares de chave e valor.

**Dicionário Python (`dict`):** estrutura de dados que armazena informações em pares de chave e valor.

**Lista Python (`list`):** estrutura que permite armazenar uma sequência de elementos.

**Estrutura hierárquica:** organização de dados em diferentes níveis, permitindo que objetos e listas sejam agrupados dentro de outras estruturas.

### Explicação dos comandos

- `response.json()`: converte o conteúdo JSON da resposta HTTP em estruturas de dados Python.
- `type(dados)`: identifica o tipo de objeto retornado.
- `dados.keys()`: apresenta as chaves existentes no dicionário principal.
- `len(dados["data"])`: contabiliza os registros armazenados na chave `data`.
- `dados["data"][:2]`: seleciona os dois primeiros registros da lista.
- `json.dumps()`: serializa um objeto Python para uma representação textual JSON.
- `indent=2`: organiza a saída com indentação de dois espaços, facilitando a leitura.
- `ensure_ascii=False`: preserva caracteres não ASCII na representação textual.

### Estrutura identificada

A resposta apresenta duas chaves principais:

- `data`: contém os registros das características ambientais.
- `count`: contém a informação de quantidade total disponibilizada pela API.

Cada registro possui campos como:

- `characteristicGroup`: grupo da característica.
- `parameterCode`: código associado ao parâmetro.
- `characteristicName`: nome da característica.
- `characteristicNameUserSupplied`: descrição detalhada fornecida para a característica.

### Resultado da execução

- Tipo de retorno: `dict`.
- Chaves principais: `data` e `count`.
- Quantidade de registros em `data`: 11.946.
- Registros inspecionados: 2.

### Conclusão

A resposta da API foi convertida com sucesso em um dicionário Python.

A inspeção permitiu identificar a estrutura hierárquica do JSON, conhecer os campos disponíveis e confirmar que a chave `data` contém os registros que serão utilizados nas próximas etapas da exploração.

A inspeção de uma amostra dos registros permite compreender a estrutura dos dados antes de realizar sua transformação para um formato tabular.

In [3]:
# Converter a resposta HTTP para objeto Python
dados = response.json()

# Inspecionar a estrutura
print("Tipo de retorno:", type(dados))
print("Chaves principais:", dados.keys())

# Quantidade de registros
print("Quantidade de registros:", len(dados["data"]))

# Visualizar os dois primeiros registros
print(json.dumps(dados["data"][:2], indent=2, ensure_ascii=False))

Tipo de retorno: <class 'dict'>
Chaves principais: dict_keys(['data', 'count'])
Quantidade de registros: 11946
[
  {
    "characteristicGroup": "Organics, Other",
    "parameterCode": "67995",
    "characteristicName": "Hydroxy-amitriptyline, 10-",
    "characteristicNameUserSupplied": "10-Hydroxy-amitriptyline, water, filtered, recoverable"
  },
  {
    "characteristicGroup": "Organics, Other",
    "parameterCode": "52417",
    "characteristicName": "1,1,1,2-Tetrachloroethane",
    "characteristicNameUserSupplied": "1,1,1,2-Tetrachloroethane, bed sediment (dry mass basis), recoverable"
  }
]


### 3.1. Resultado e análise da resposta JSON

A execução apresentou os seguintes resultados:

| Indicador | Resultado |
|---|---|
| Tipo do objeto | `dict` |
| Chaves principais | `data`, `count` |
| Registros na lista `data` | 11.946 |
| Registros visualizados | 2 |

#### Análise da estrutura

A resposta apresenta uma estrutura hierárquica:

- O objeto principal é um dicionário Python.
- A chave `data` contém uma lista de registros.
- Cada elemento dessa lista representa uma característica ambiental.
- Cada característica possui atributos identificadores e descritivos.

#### Observação técnica

A quantidade de 11.946 foi obtida por meio de `len(dados["data"])`, ou seja, representa a quantidade de elementos presentes na lista retornada.

Embora a resposta também apresente a chave `count`, seu conteúdo ainda não foi inspecionado individualmente.

#### Conclusão

A estrutura JSON foi interpretada com sucesso e os registros estão disponíveis em estruturas nativas do Python.

Essa etapa estabelece a base para transformar os dados semiestruturados em uma estrutura tabular utilizando Pandas.

## 4. Conversão dos registros JSON para DataFrame

### Objetivo

Converter a lista de registros retornada pela API USGS em um DataFrame Pandas, permitindo visualizar os dados em formato tabular e investigar suas dimensões, colunas e tipos de dados.

### Justificativa

Após identificar a estrutura hierárquica da resposta JSON, precisamos organizar os registros em uma estrutura adequada para análise e transformação.

O Pandas disponibiliza o objeto DataFrame, que permite trabalhar com dados organizados em linhas e colunas, facilitando operações de filtragem, agrupamento, tratamento e análise.

### Conceitos técnicos

**DataFrame:** estrutura bidimensional do Pandas, organizada em linhas e colunas.

**Shape:** propriedade que retorna as dimensões do DataFrame, representadas por uma tupla (linhas, colunas).

**Columns:** propriedade que permite identificar os nomes das colunas existentes.

**Dtypes:** propriedade que apresenta os tipos de dados de cada coluna.

**Head:** método utilizado para visualizar os primeiros registros de um DataFrame.

### Explicação dos comandos

- `pd.DataFrame(dados["data"])`: converte a lista de registros JSON em um DataFrame.
- `.shape`: retorna a quantidade de linhas e colunas.
- `.columns`: retorna os nomes das colunas.
- `.tolist()`: converte os nomes das colunas em uma lista Python.
- `.dtypes`: identifica os tipos de dados de cada coluna.
- `.head()`: apresenta os cinco primeiros registros.
- `display()`: apresenta o objeto em formato visual no ambiente Jupyter Notebook.

### Resultado esperado

Um DataFrame contendo os registros disponibilizados pela API, acompanhado de informações sobre dimensões, colunas e tipos de dados.

In [4]:
# Converter a lista de registros em DataFrame
df_caracteristicas = pd.DataFrame(dados["data"])

# Inspecionar dimensões
print("Dimensões:", df_caracteristicas.shape)

# Identificar colunas
print("\nColunas:")
print(df_caracteristicas.columns.tolist())

# Identificar tipos de dados
print("\nTipos de dados:")
print(df_caracteristicas.dtypes)

# Visualizar registros
display(df_caracteristicas.head())

Dimensões: (11946, 4)

Colunas:
['characteristicGroup', 'parameterCode', 'characteristicName', 'characteristicNameUserSupplied']

Tipos de dados:
characteristicGroup               str
parameterCode                     str
characteristicName                str
characteristicNameUserSupplied    str
dtype: object


,characteristicGroup,parameterCode,characteristicName,characteristicNameUserSupplied
0,"Organics, Other",67995,"Hydroxy-amitriptyline, 10-","10-Hydroxy-amitriptyline, water, filtered, rec..."
1,"Organics, Other",52417,"1,1,1,2-Tetrachloroethane","1,1,1,2-Tetrachloroethane, bed sediment (dry m..."
2,"Organics, Other",30089,"1,1,1,2-Tetrachloroethane","1,1,1,2-Tetrachloroethane, soil (dry mass basi..."
3,"Organics, Other",62235,"1,1,1,2-Tetrachloroethane","1,1,1,2-Tetrachloroethane, soil (dry mass basi..."
4,"Organics, Other",77562,"1,1,1,2-Tetrachloroethane","1,1,1,2-Tetrachloroethane, water, unfiltered, ..."


### 4.1. Resultado e análise estrutural

A conversão dos registros JSON para DataFrame foi realizada com sucesso.

**Dimensões identificadas:** 11.946 linhas e 4 colunas.

**Colunas identificadas:**
- `characteristicGroup`
- `parameterCode`
- `characteristicName`
- `characteristicNameUserSupplied`

**Tipos de dados:** todas as quatro colunas foram identificadas como texto (`str`).

#### Análise técnica

A estrutura tabular permite trabalhar individualmente com cada atributo dos registros.

A identificação dos tipos de dados também é importante para evitar conversões inadequadas durante as próximas etapas de transformação.

O campo `parameterCode` foi mantido como texto, preservando sua representação como código identificador.

#### Conclusão

O DataFrame `df_caracteristicas` foi criado com sucesso e está disponível para as próximas etapas de exploração, filtragem e análise das características ambientais disponibilizadas pela API USGS.

## 5. Agrupamento e contagem de características por grupo

### Objetivo

Identificar a quantidade de características ambientais existentes em cada grupo, utilizando operações de agrupamento e agregação do Pandas.

### Justificativa

Após transformar os registros JSON em um DataFrame, podemos começar a explorar a distribuição das características ambientais.

O agrupamento permite organizar os registros de acordo com valores comuns de uma coluna e calcular indicadores para cada grupo.

Neste caso, queremos identificar quais grupos possuem a maior quantidade de características cadastradas na API USGS.

### Conceitos técnicos

**Group By:** operação que agrupa registros de acordo com uma ou mais colunas.

**Size:** método que contabiliza a quantidade de registros existentes em cada grupo, incluindo linhas com outros campos nulos.

**Sort Values:** método utilizado para ordenar os resultados de acordo com os valores de uma coluna ou série.

**Ascending:** parâmetro que define a ordem da classificação. Quando configurado como `False`, apresenta os maiores valores primeiro.

### Explicação dos comandos

- `df_caracteristicas.groupby("characteristicGroup")`: agrupa os registros pela coluna de grupo da característica.
- `.size()`: contabiliza a quantidade de registros em cada grupo.
- `.sort_values(ascending=False)`: ordena os grupos da maior para a menor quantidade.
- `display(grupos.head(15))`: apresenta os 15 primeiros resultados.

### Resultado esperado

Uma relação dos grupos de características ambientais, ordenada pela quantidade de registros existentes em cada grupo.

In [5]:
# Quantidade de características por grupo
grupos = (
    df_caracteristicas
    .groupby("characteristicGroup")
    .size()
    .sort_values(ascending=False)
)

display(grupos.head(15))

characteristicGroup
Organics, Other                  2269
Organics, Pesticide              2084
Population/Community             1999
Inorganics, Minor, Metals        1004
Organics, PCBs                    985
Information                       663
Organics, PFAS                    464
Physical                          460
Sediment                          408
Microbiological                   270
Radiochemical                     269
Inorganics, Major, Non-metals     256
Inorganics, Minor, Non-metals     184
Nutrient                          171
Inorganics, Major, Metals         152
dtype: int64

### 5.1. Resultado e análise do agrupamento

O agrupamento dos registros pela coluna `characteristicGroup` foi executado com sucesso.

A operação identificou a quantidade de registros em cada grupo e organizou os resultados em ordem decrescente.

#### Principais resultados

Os três grupos com maior quantidade de registros foram:

- Organics, Other: 2.269 registros.
- Organics, Pesticide: 2.084 registros.
- Population/Community: 1.999 registros.

#### Análise técnica

A operação `groupby()` combinada com `size()` permitiu realizar uma agregação dos registros sem necessidade de percorrer individualmente as linhas do DataFrame.

O resultado foi armazenado em uma Series Pandas e ordenado utilizando `sort_values()`.

É importante destacar que a contagem representa o número de registros por grupo, não a quantidade de características únicas.

#### Conclusão

A análise permitiu identificar a distribuição dos registros entre os grupos de características ambientais disponibilizados pela API USGS.

Essa exploração inicial fornece uma visão da composição do catálogo e prepara os dados para análises mais específicas sobre nomes de características, códigos de parâmetros e possíveis duplicidades.

## 6. Seleção e filtragem de características de interesse

### Objetivo

Selecionar os grupos de características ambientais considerados relevantes para a análise da qualidade da água e filtrar os registros correspondentes no DataFrame original.

### Justificativa

O catálogo da API USGS disponibiliza uma grande variedade de características ambientais, mas nem todas são relevantes para o escopo deste projeto.

Para direcionar a investigação, definimos uma lista de grupos relacionados à qualidade da água e utilizamos essa lista como critério de seleção.

Essa abordagem permite reduzir o conjunto de dados e concentrar as próximas análises nas características de maior interesse para o projeto.

### Grupos selecionados

- Physical
- Microbiological
- Nutrient
- Inorganics, Major, Non-metals
- Inorganics, Major, Metals
- Radiochemical

### Conceitos técnicos

**Lista Python (`list`):** estrutura utilizada para armazenar uma coleção ordenada de elementos.

**Filtro booleano:** operação que seleciona registros de acordo com condições que retornam `True` ou `False`.

**isin():** método do Pandas que verifica se os valores de uma coluna pertencem a uma lista ou coleção de referência.

**copy():** método que cria uma cópia independente dos dados selecionados, evitando alterações acidentais no DataFrame original.

**sort_values():** método utilizado para ordenar os registros por uma ou mais colunas.

### Explicação dos comandos

- `grupos_interesse = [...]`: define os grupos que serão considerados na análise.
- `df_caracteristicas["characteristicGroup"]`: acessa a coluna utilizada como critério de filtragem.
- `.isin(grupos_interesse)`: verifica quais registros pertencem aos grupos selecionados.
- `df_caracteristicas[condição]`: retorna somente os registros que atendem ao filtro.
- `.copy()`: cria uma cópia independente do subconjunto selecionado.
- `len(df_selecao)`: contabiliza os registros selecionados.
- `df_selecao[[...]]`: seleciona somente as colunas necessárias para visualização.
- `.sort_values(["characteristicGroup", "characteristicName"])`: ordena os registros pelo grupo e pelo nome da característica.
- `display()`: apresenta o resultado no notebook.

In [6]:
# Grupos de interesse para qualidade da água

grupos_interesse = [
    "Physical",
    "Microbiological",
    "Nutrient",
    "Inorganics, Major, Non-metals",
    "Inorganics, Major, Metals",
    "Radiochemical"
]

# Filtrar características
df_selecao = df_caracteristicas[
    df_caracteristicas["characteristicGroup"].isin(grupos_interesse)
].copy()

# Visualizar resultado
print("Total de características selecionadas:", len(df_selecao))

display(
    df_selecao[
        [
            "characteristicGroup",
            "parameterCode",
            "characteristicName"
        ]
    ].sort_values(
        ["characteristicGroup", "characteristicName"]
    )
)

Total de características selecionadas: 1578


,characteristicGroup,parameterCode,characteristicName
2607,"Inorganics, Major, Metals",00910,Calcium
2608,"Inorganics, Major, Metals",62456,Calcium
2609,"Inorganics, Major, Metals",30239,Calcium
2610,"Inorganics, Major, Metals",52713,Calcium
2611,"Inorganics, Major, Metals",01476,Calcium
...,...,...,...
11843,Radiochemical,82889,Xenon-136
11844,Radiochemical,85634,Xenon-136
11891,Radiochemical,29301,Zinc-65
11926,Radiochemical,27701,Zirconium-95


### 6.1. Resultado e análise da seleção

A filtragem dos registros foi executada com sucesso.

**Resultado:** 1.578 registros selecionados a partir dos 11.946 registros originalmente disponíveis.

#### Critérios utilizados

Foram selecionados seis grupos de características ambientais relacionados ao escopo da análise de qualidade da água.

A filtragem foi realizada utilizando o método `isin()`, que verifica a correspondência dos valores da coluna `characteristicGroup` com os elementos da lista `grupos_interesse`.

#### Análise técnica

O resultado foi armazenado em um novo DataFrame denominado `df_selecao`.

A utilização de `.copy()` cria uma cópia independente do subconjunto selecionado, permitindo realizar transformações posteriores sem modificar diretamente o DataFrame original.

Para facilitar a inspeção, foram apresentadas três colunas e os registros foram ordenados alfabeticamente pelo grupo e pelo nome da característica.

#### Conclusão

A seleção reduziu o conjunto de dados de 11.946 para 1.578 registros, concentrando a investigação em seis grupos considerados relevantes para o projeto.

O DataFrame filtrado será utilizado nas próximas etapas para investigar características específicas, códigos de parâmetros e possíveis duplicidades.

## 7. Pesquisa e identificação de parâmetros de qualidade da água

### Objetivo

Identificar registros relacionados a parâmetros de interesse para a análise da qualidade da água, utilizando palavras-chave e expressões regulares para pesquisar em diferentes colunas do DataFrame filtrado.

### Justificativa

Após selecionar os grupos de características ambientais relevantes, precisamos identificar quais registros correspondem aos parâmetros que pretendemos investigar.

Como os nomes e as descrições das características podem apresentar variações textuais, utilizaremos expressões regulares para ampliar a capacidade de pesquisa.

A pesquisa será realizada nas colunas `characteristicName` e `characteristicNameUserSupplied`, considerando tanto o nome da característica quanto sua descrição detalhada.

### Termos pesquisados

- pH
- dissolved oxygen
- temperature
- conductivity
- turbidity
- nitrate
- phosphorus
- e. coli

### Conceitos técnicos

**Expressões regulares (RegEx):** padrões utilizados para localizar, identificar e manipular sequências de caracteres em textos.

**re.escape():** função do módulo `re` que transforma caracteres especiais em caracteres literais, evitando que sejam interpretados como operadores de expressão regular.

**re.IGNORECASE / case=False:** permite realizar pesquisas sem diferenciar letras maiúsculas e minúsculas.

**str.contains():** método do Pandas que verifica se determinado padrão textual está presente em cada elemento de uma coluna.

**Máscara booleana:** estrutura de valores `True` e `False` utilizada para selecionar registros.

**Operador OR (`|`):** combina condições, permitindo selecionar registros que atendam a pelo menos uma delas.

**nunique():** método que contabiliza a quantidade de valores distintos de uma coluna, desconsiderando valores ausentes por padrão.

### Explicação dos comandos

- `termos = [...]`: define as palavras-chave utilizadas na pesquisa.
- `padroes = [...]`: inicializa a lista de padrões RegEx.
- `(?<![A-Za-z])pH(?![A-Za-z])`: procura pH sem que existam letras imediatamente antes ou depois.
- `termos[1:]`: seleciona os elementos da lista a partir do segundo, excluindo pH.
- `re.escape(termo)`: protege caracteres especiais presentes nos termos pesquisados.
- `padroes.extend()`: acrescenta novos padrões à lista existente.
- `"|".join(padroes)`: combina os padrões utilizando o operador OR das expressões regulares.
- `.str.contains()`: pesquisa o padrão nas colunas selecionadas.
- `case=False`: ignora diferenças entre letras maiúsculas e minúsculas.
- `na=False`: considera valores ausentes como não correspondentes à pesquisa.
- `df_selecao.loc[mascara]`: seleciona os registros que atendem aos critérios.
- `.copy()`: cria uma cópia independente dos registros selecionados.
- `.nunique()`: identifica a quantidade de códigos distintos.

### Resultado esperado

Um novo DataFrame contendo os registros relacionados aos termos pesquisados, acompanhado da quantidade total de registros e da quantidade de códigos de parâmetros distintos.

In [7]:
# Pesquisa de parâmetros de qualidade da água

termos = [
    "pH",
    "dissolved oxygen",
    "temperature",
    "conductivity",
    "turbidity",
    "nitrate",
    "phosphorus",
    "e. coli"
]

# Construção dos padrões
padroes = [
    r"(?<![A-Za-z])pH(?![A-Za-z])"
]

padroes.extend(
    rf"(?<!\w){re.escape(termo)}(?!\w)"
    for termo in termos[1:]
)

padrao = "|".join(padroes)

# Pesquisa nas colunas
mascara = (
    df_selecao["characteristicName"].str.contains(
        padrao, case=False, na=False
    )
    |
    df_selecao["characteristicNameUserSupplied"].str.contains(
        padrao, case=False, na=False
    )
)

# Criação do DataFrame filtrado
df_parametros = df_selecao.loc[mascara].copy()

print("Total de registros:", len(df_parametros))
print("Códigos distintos:", df_parametros["parameterCode"].nunique())

display(df_parametros.head(30))

Total de registros: 240
Códigos distintos: 221


,characteristicGroup,parameterCode,characteristicName,characteristicNameUserSupplied
1097,"Inorganics, Major, Non-metals",00410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1098,"Inorganics, Major, Non-metals",95410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1099,"Inorganics, Major, Non-metals",00417,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1100,"Inorganics, Major, Non-metals",90410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1105,"Inorganics, Major, Non-metals",00411,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1106,"Inorganics, Major, Non-metals",00415,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1217,"Inorganics, Major, Non-metals",39036,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1218,"Inorganics, Major, Non-metals",00418,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1219,"Inorganics, Major, Non-metals",00421,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1220,"Inorganics, Major, Non-metals",29801,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."


### 7.1. Resultado e análise da pesquisa de parâmetros

A pesquisa textual foi executada com sucesso nas colunas `characteristicName` e `characteristicNameUserSupplied`.

**Resultados obtidos:**

- Registros encontrados: 240.
- Códigos distintos: 221.
- Termos pesquisados: 8.

#### Análise técnica

A utilização de expressões regulares permitiu combinar diferentes critérios de pesquisa em um único padrão.

A máscara booleana foi construída considerando correspondências em duas colunas, utilizando o operador OR.

Os registros encontrados foram armazenados no DataFrame `df_parametros`, preservando a estrutura original dos dados.

#### Observação

A diferença entre registros encontrados e códigos distintos indica a necessidade de investigar possíveis repetições de códigos e valores ausentes.

A quantidade de códigos distintos não deve ser interpretada como quantidade definitiva de parâmetros de qualidade da água.

#### Conclusão

A pesquisa identificou 240 registros relacionados aos termos definidos, contendo 221 códigos distintos.

O resultado fornece uma primeira seleção de características para investigação mais detalhada, permitindo avançar para a análise de correspondências, duplicidades e consistência dos códigos de parâmetros.

## 8. Diagnóstico individual das correspondências

### Objetivo

Investigar individualmente a quantidade de registros encontrados para cada termo utilizado na pesquisa de parâmetros de qualidade da água.

### Justificativa

Na etapa anterior, os oito termos foram combinados em uma única expressão regular, resultando em 240 registros selecionados.

Entretanto, essa abordagem não permite identificar diretamente quantas correspondências foram encontradas para cada termo.

Para solucionar essa limitação, será executado um diagnóstico individual, percorrendo a lista de termos e aplicando cada padrão de pesquisa separadamente.

### Estratégia

Para cada termo:

1. Verificar se o termo corresponde a pH.
2. Construir o padrão de expressão regular apropriado.
3. Pesquisar nas colunas `characteristicName` e `characteristicNameUserSupplied`.
4. Combinar as condições utilizando o operador OR.
5. Contabilizar as correspondências encontradas.
6. Exibir o resultado individualmente.

### Conceitos técnicos

**Laço `for`:** estrutura de repetição que percorre os elementos de uma coleção.

**Condicional `if/else`:** estrutura utilizada para executar diferentes instruções conforme uma condição.

**Máscara booleana:** conjunto de valores `True` e `False` que representa quais registros atendem aos critérios de pesquisa.

**Método `sum()`:** soma os valores da máscara booleana. Como `True` equivale a 1 e `False` equivale a 0, o resultado representa a quantidade de correspondências.

**Interpolação de strings (`f-string`):** permite inserir variáveis diretamente em textos utilizando o prefixo `f`.

### Resultado esperado

Uma listagem contendo a quantidade de correspondências encontradas individualmente para cada um dos oito termos pesquisados.

Essa análise também permite identificar possíveis sobreposições entre os critérios de pesquisa.

In [8]:
# Diagnóstico das correspondências

for termo in termos:

    if termo == "pH":
        padrao_teste = r"(?<![A-Za-z])pH(?![A-Za-z])"
    else:
        padrao_teste = rf"(?<!\w){re.escape(termo)}(?!\w)"

    resultado = (
        df_selecao["characteristicName"].str.contains(
            padrao_teste, case=False, na=False
        )
        |
        df_selecao["characteristicNameUserSupplied"].str.contains(
            padrao_teste, case=False, na=False
        )
    )

    print(f"{termo}: {resultado.sum()} correspondências")

pH: 46 correspondências
dissolved oxygen: 11 correspondências
temperature: 39 correspondências
conductivity: 0 correspondências
turbidity: 26 correspondências
nitrate: 54 correspondências
phosphorus: 61 correspondências
e. coli: 4 correspondências


### 8.1. Resultado e análise do diagnóstico

O diagnóstico individual dos oito termos foi executado com sucesso.

#### Resultados obtidos

| Termo | Correspondências |
|---|---:|
| pH | 46 |
| dissolved oxygen | 11 |
| temperature | 39 |
| conductivity | 0 |
| turbidity | 26 |
| nitrate | 54 |
| phosphorus | 61 |
| e. coli | 4 |

A soma das correspondências individuais foi de 241, enquanto a pesquisa combinada da etapa anterior identificou 240 registros.

#### Análise técnica

A diferença entre os resultados indica uma possível sobreposição entre os critérios de pesquisa.

Isso ocorre porque um mesmo registro pode conter mais de um termo pesquisado. Na contagem individual, ele participa da contabilização de cada termo correspondente. Na pesquisa combinada, entretanto, o registro é selecionado apenas uma vez.

O diagnóstico também identificou ausência de correspondências para o termo `conductivity`, indicando a necessidade de investigar possíveis variações de nomenclatura.

#### Conclusão

A análise individual permitiu identificar a distribuição das correspondências entre os termos pesquisados e evidenciou a necessidade de investigar sobreposições e possíveis diferenças de nomenclatura.

Esses resultados serão utilizados para orientar as próximas etapas de validação e preparação dos dados.

## 9. Inspeção dos registros de Alcalinidade (Alkalinity)

### Objetivo

Investigar os registros identificados com o nome de característica `Alkalinity`, verificando os códigos de parâmetros associados e suas respectivas descrições.

### Justificativa

Durante a pesquisa de parâmetros da etapa anterior, foram encontrados registros que compartilham o mesmo nome de característica, mas apresentam códigos de parâmetros diferentes.

Para compreender essa ocorrência, será realizada uma inspeção específica dos registros de alcalinidade.

A análise permitirá verificar se os diferentes códigos representam variações de descrição, condições de medição ou outras classificações disponibilizadas pela API USGS.

### Estratégia

1. Filtrar o DataFrame `df_parametros` utilizando a coluna `characteristicName`.
2. Selecionar os registros cujo nome seja exatamente `Alkalinity`.
3. Criar um novo DataFrame independente.
4. Exibir os códigos e as descrições associadas à característica.

### Conceitos técnicos

**Comparação de igualdade (`==`):** verifica se o valor de uma coluna corresponde exatamente ao valor especificado.

**Filtro booleano:** seleciona os registros que atendem à condição estabelecida.

**`.loc[]`:** permite acessar e selecionar linhas e colunas de um DataFrame utilizando rótulos e condições.

**`.copy()`:** cria uma cópia independente dos registros selecionados.

**Seleção de colunas:** permite restringir a visualização aos campos relevantes para a investigação.

### Resultado esperado

Uma relação dos registros de alcalinidade contendo seus códigos de parâmetros e descrições detalhadas, permitindo investigar a existência de diferentes representações da mesma característica.

In [9]:
# Inspecionar registros de Alkalinity

df_diagnostico = df_parametros[
    df_parametros["characteristicName"] == "Alkalinity"
].copy()

display(
    df_diagnostico[
        [
            "parameterCode",
            "characteristicName",
            "characteristicNameUserSupplied"
        ]
    ]
)

,parameterCode,characteristicName,characteristicNameUserSupplied
1097,00410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1098,95410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1099,00417,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1100,90410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1105,00411,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1106,00415,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1217,39036,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1218,00418,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1219,00421,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1220,29801,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."


### 9.1. Resultado e análise dos registros de Alkalinity

A inspeção dos registros de alcalinidade foi executada com sucesso.

**Resultado:** foram identificados 10 registros com o nome de característica `Alkalinity`.

#### Principais observações

Todos os registros apresentam o mesmo nome padronizado, mas estão associados a diferentes códigos de parâmetros e descrições detalhadas.

As descrições apresentam referências a diferentes formas de representação da alcalinidade, incluindo capacidade de neutralização ácida (ANC), carbonato de cálcio e condições de filtragem da água.

#### Análise técnica

A investigação demonstra que a coluna `characteristicName`, isoladamente, não é suficiente para determinar a equivalência entre parâmetros.

É necessário considerar também o código do parâmetro e sua descrição detalhada.

Outro cuidado importante é preservar os códigos como texto, mantendo zeros à esquerda e a representação original dos identificadores.

#### Conclusão

A inspeção permitiu identificar diferentes códigos associados à característica `Alkalinity`.

Os resultados demonstram a necessidade de investigar a semântica dos códigos e suas descrições antes de estabelecer regras de padronização, agrupamento ou eliminação de registros.

Essa análise contribui para a definição de critérios de qualidade e consistência dos dados no pipeline ETL.

## 10. Investigação das correspondências de Alkalinity

### Objetivo

Investigar quais termos de pesquisa identificaram os registros de alcalinidade selecionados anteriormente, verificando em qual coluna textual ocorreram as correspondências.

### Justificativa

Na Célula 09, foram identificados dez registros com `characteristicName = Alkalinity`.

Entretanto, esses registros foram selecionados durante a pesquisa de parâmetros da Célula 07, que utiliza diferentes termos relacionados à qualidade da água.

Precisamos identificar por que registros de alcalinidade foram incluídos na seleção e verificar se a correspondência ocorreu no nome padronizado ou na descrição detalhada.

Essa investigação ajuda a identificar falsos positivos produzidos por pesquisas textuais e a avaliar a precisão dos critérios de seleção.

### Estratégia

1. Percorrer simultaneamente a lista de termos e seus respectivos padrões RegEx.
2. Verificar cada termo nas colunas `characteristicName` e `characteristicNameUserSupplied`.
3. Construir uma máscara booleana para cada pesquisa.
4. Verificar se existem correspondências utilizando `any()`.
5. Contabilizar as ocorrências utilizando `sum()`.
6. Exibir os registros encontrados e seus respectivos códigos.

### Conceitos técnicos

**zip():** função Python que combina elementos de duas ou mais coleções, permitindo percorrê-los simultaneamente.

**for aninhado:** estrutura de repetição dentro de outra estrutura de repetição.

**any():** verifica se pelo menos um elemento de uma coleção é verdadeiro.

**sum():** contabiliza os valores verdadeiros de uma máscara booleana.

**to_string():** converte um DataFrame em representação textual, permitindo controlar a apresentação dos índices.

**Regex:** mecanismo utilizado para identificar padrões de texto.

### Resultado esperado

Identificar quais termos produziram correspondências nos registros de alcalinidade e em quais colunas essas correspondências ocorreram.

In [10]:
# Investigar correspondências de Alkalinity

for termo, padrao_teste in zip(termos, padroes):

    for coluna in [
        "characteristicName",
        "characteristicNameUserSupplied"
    ]:

        resultado = df_diagnostico[coluna].str.contains(
            padrao_teste,
            case=False,
            na=False,
            regex=True
        )

        if resultado.any():

            print(f"\nTermo: {termo}")
            print(f"Coluna: {coluna}")
            print("Quantidade:", resultado.sum())

            display(
                df_diagnostico.loc[
                    resultado,
                    [coluna, "parameterCode"]
                ].to_string(index=False)
            )


Termo: pH
Coluna: characteristicNameUserSupplied
Quantidade: 10


'                                                                                                characteristicNameUserSupplied parameterCode\n       Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, fixed endpoint (EP) (pH 4.5) titration, field         00410\n  Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, fixed endpoint (EP) (pH 4.5) titration, laboratory         95410\n  Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, fixed endpoint (EP) (pH 4.5) titration, laboratory         00417\n  Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, fixed endpoint (EP) (pH 4.5) titration, laboratory         90410\n  Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, methyl orange endpoint (EP) (pH 3.1-4.4) titration         00411\nAcid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, phenolphthalein endpoint (EP) (pH 8.5-9.0) titration         00415\n     

### 10.1. Resultado e análise das correspondências de Alkalinity

A investigação das correspondências foi executada com sucesso.

#### Resultado identificado

Foram encontradas dez correspondências para o termo `pH`, todas na coluna `characteristicNameUserSupplied`.

Não foram identificadas correspondências para os demais termos pesquisados nos registros de alcalinidade.

#### Análise técnica

A investigação demonstrou que os registros de `Alkalinity` foram incluídos na seleção anterior porque suas descrições detalhadas contêm referências a procedimentos de titulação que utilizam valores de pH como pontos finais.

Isso caracteriza um possível falso positivo na pesquisa textual.

A característica principal continua sendo alcalinidade, embora sua descrição contenha referências a pH.

#### Conclusão

A análise evidenciou uma limitação da identificação de parâmetros baseada exclusivamente em correspondências textuais.

Para aumentar a precisão da seleção, será necessário considerar o nome padronizado da característica, os códigos de parâmetros e as descrições detalhadas em conjunto.

Essa investigação reforça a importância da validação semântica dos dados antes de sua utilização em transformações e análises posteriores.

## 11. Pesquisa refinada de parâmetros

### Objetivo

Refinar a identificação dos parâmetros relacionados à qualidade da água, utilizando exclusivamente a coluna `characteristicName` como critério de pesquisa.

### Justificativa

Na investigação anterior, identificamos que a pesquisa realizada também na coluna `characteristicNameUserSupplied` produzia correspondências que não representavam necessariamente a característica principal do registro.

Um exemplo foi a identificação de registros de `Alkalinity` por conterem referências a pH nas descrições de procedimentos de titulação.

Para reduzir esse tipo de ocorrência, a estratégia será modificada.

A pesquisa passará a considerar somente os nomes padronizados das características, evitando que referências secundárias presentes nas descrições detalhadas determinem a seleção.

### Estratégia

1. Definir expressões regulares para os oito parâmetros de interesse.
2. Considerar variações de nomenclatura para condutividade.
3. Construir uma expressão regular combinada utilizando o operador OR.
4. Pesquisar exclusivamente na coluna `characteristicName`.
5. Criar uma cópia independente dos registros selecionados.
6. Contabilizar os registros encontrados e os códigos distintos.
7. Exibir uma amostra de até 50 registros.

### Conceitos técnicos

**Expressões regulares (RegEx):** permitem identificar padrões específicos em textos.

**Alternância (`|`):** permite pesquisar diferentes padrões utilizando uma única expressão regular.

**str.contains():** identifica valores que contêm determinado padrão textual.

**case=False:** torna a pesquisa independente de letras maiúsculas ou minúsculas.

**na=False:** trata valores ausentes como não correspondentes.

**nunique():** contabiliza a quantidade de valores distintos de uma coluna, desconsiderando valores ausentes por padrão.

**head(50):** retorna os primeiros 50 registros do DataFrame.

### Resultado esperado

Uma seleção refinada dos parâmetros de interesse, com menor interferência de palavras encontradas apenas nas descrições complementares.

In [11]:
# Pesquisa refinada de parâmetros

padroes_nome = [
    r"(?<![A-Za-z])pH(?![A-Za-z])",
    r"dissolved oxygen",
    r"temperature",
    r"conductivity|specific conductance",
    r"turbidity",
    r"nitrate",
    r"phosphorus",
    r"e\. coli"
]

padrao_nome = "|".join(padroes_nome)

# Pesquisar somente no nome da característica
mascara = df_selecao["characteristicName"].str.contains(
    padrao_nome,
    case=False,
    na=False,
    regex=True
)

# Criar DataFrame independente
df_parametros = df_selecao.loc[mascara].copy()

# Exibir resultados
print("Total de registros:", len(df_parametros))
print(
    "Códigos distintos:",
    df_parametros["parameterCode"].nunique()
)

display(
    df_parametros[
        [
            "characteristicGroup",
            "parameterCode",
            "characteristicName",
            "characteristicNameUserSupplied"
        ]
    ].head(50)
)

Total de registros: 176
Códigos distintos: 158


,characteristicGroup,parameterCode,characteristicName,characteristicNameUserSupplied
4211,"Inorganics, Major, Non-metals",90300,Dissolved oxygen (DO),"Dissolved oxygen (DO), area-weighted mean, wat..."
4212,"Inorganics, Major, Non-metals",00301,Dissolved oxygen saturation,"Dissolved oxygen (DO) saturation, water, unfil..."
4213,"Inorganics, Major, Non-metals",NaN,Dissolved oxygen saturation,"Dissolved oxygen (DO) saturation, water, unfil..."
4214,"Inorganics, Major, Non-metals",72210,Dissolved oxygen saturation,"Dissolved oxygen (DO) saturation, water, unfil..."
4215,"Inorganics, Major, Non-metals",85550,Dissolved oxygen (DO),"Dissolved oxygen (DO) (volume basis), dissolve..."
4216,"Inorganics, Major, Non-metals",NaN,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered"
4217,"Inorganics, Major, Non-metals",00300,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered"
4218,"Inorganics, Major, Non-metals",NaN,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered, labo..."
4219,"Inorganics, Major, Non-metals",62971,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered, labo..."
4220,"Inorganics, Major, Non-metals",99985,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered, mid-..."


### 11.1. Resultado e análise da pesquisa refinada

A pesquisa refinada dos parâmetros foi executada com sucesso.

#### Resultados obtidos

- Total de registros selecionados: 176.
- Códigos distintos: 158.

#### Alteração da estratégia

A principal alteração foi restringir a pesquisa à coluna `characteristicName`, deixando de utilizar a descrição detalhada como critério de seleção.

Essa mudança foi motivada pela identificação de possíveis falsos positivos na investigação anterior, especialmente registros de alcalinidade identificados pela presença de referências a pH em suas descrições.

Também foram incorporadas variações de nomenclatura para condutividade, contemplando `conductivity` e `specific conductance`.

#### Análise técnica

A pesquisa refinada reduziu o resultado de 240 para 176 registros.

A amostra apresenta diferentes características relacionadas aos grupos físicos, microbiológicos, nutricionais e inorgânicos.

Também foram identificados registros sem código de parâmetro (`NaN`), indicando a necessidade de uma investigação adicional sobre a completude dos identificadores.

#### Conclusão

A nova estratégia tornou o critério de seleção mais restritivo e reduziu a influência de descrições complementares sobre a identificação dos parâmetros.

Entretanto, os resultados ainda precisam passar por validações de integridade, unicidade e correspondência semântica antes de serem considerados um catálogo definitivo de parâmetros de qualidade da água.

## 12. Análise de cardinalidade dos parâmetros

### Objetivo

Avaliar a cardinalidade, a completude e a distribuição dos parâmetros identificados na pesquisa refinada, verificando a quantidade de registros, nomes distintos, códigos distintos e identificadores ausentes.

### Justificativa

Após a seleção refinada dos 176 registros, é necessário compreender melhor a estrutura e a qualidade do conjunto de dados.

A existência de diferentes registros associados à mesma característica pode indicar variações de parâmetros, condições de medição ou representações distintas.

Além disso, foram identificados códigos ausentes, tornando necessária uma análise da completude dos identificadores.

### Estratégia

1. Contabilizar o total de registros.
2. Identificar a quantidade de nomes distintos em `characteristicName`.
3. Identificar a quantidade de códigos distintos em `parameterCode`.
4. Contabilizar os códigos ausentes.
5. Agrupar os registros por grupo e nome da característica.
6. Calcular métricas de quantidade de registros, códigos distintos e códigos ausentes.
7. Ordenar o resultado pela quantidade de registros, em ordem decrescente.

### Conceitos técnicos

**Cardinalidade:** representa a quantidade de valores distintos existentes em uma variável ou coluna.

**nunique():** contabiliza valores distintos, desconsiderando valores ausentes por padrão.

**isna():** identifica valores ausentes, retornando uma máscara booleana.

**groupby():** agrupa registros de acordo com uma ou mais colunas.

**dropna=False:** permite manter grupos cujas chaves possuem valores ausentes.

**agg():** executa múltiplas operações de agregação sobre os grupos.

**size:** contabiliza a quantidade de linhas de cada grupo.

**lambda:** permite definir uma função anônima para executar uma operação específica.

**reset_index():** transforma os índices dos grupos em colunas do DataFrame.

**sort_values():** ordena os registros de acordo com uma ou mais colunas.

### Resultado esperado

Um diagnóstico quantitativo dos parâmetros selecionados, permitindo identificar características com maior quantidade de registros, variações de códigos e problemas de completude dos identificadores.

In [12]:
# Análise de cardinalidade dos parâmetros

print("Registros:", len(df_parametros))

print(
    "Nomes distintos:",
    df_parametros["characteristicName"].nunique()
)

print(
    "Códigos distintos:",
    df_parametros["parameterCode"].nunique()
)

print(
    "Códigos ausentes:",
    df_parametros["parameterCode"].isna().sum()
)

# Distribuição por nome da característica
resumo_parametros = (
    df_parametros
    .groupby(
        ["characteristicGroup", "characteristicName"],
        dropna=False
    )
    .agg(
        registros=("parameterCode", "size"),
        codigos_distintos=("parameterCode", "nunique"),
        codigos_ausentes=("parameterCode", lambda x: x.isna().sum())
    )
    .reset_index()
    .sort_values("registros", ascending=False)
)

display(resumo_parametros)

Registros: 176
Nomes distintos: 24
Códigos distintos: 158
Códigos ausentes: 14


,characteristicGroup,characteristicName,registros,codigos_distintos,codigos_ausentes
12,Nutrient,Phosphorus,40,39,1
21,Physical,Turbidity,25,21,2
9,Nutrient,Nitrate,21,21,0
23,Physical,pH,18,15,3
13,Physical,Specific conductance,15,13,2
0,"Inorganics, Major, Non-metals",Dissolved oxygen (DO),8,6,2
7,Nutrient,Inorganic nitrogen (nitrate and nitrite),8,7,1
19,Physical,"Temperature, water",6,5,1
14,Physical,Temperature,5,5,0
15,Physical,"Temperature, air",4,3,1


### 12.1. Resultado e análise da cardinalidade dos parâmetros

A análise de cardinalidade foi executada com sucesso.

#### Indicadores gerais

| Indicador | Resultado |
|---|---:|
| Registros analisados | 176 |
| Nomes distintos | 24 |
| Códigos distintos | 158 |
| Códigos ausentes | 14 |

A completude dos códigos de parâmetros foi de aproximadamente 92,05%.

#### Principais características identificadas

A característica `Phosphorus` apresentou a maior quantidade de registros, totalizando 40 ocorrências.

Em seguida, destacaram-se `Turbidity`, com 25 registros, e `Nitrate`, com 21 registros.

A análise também identificou diferenças entre a quantidade de registros e a quantidade de códigos distintos, evidenciando a existência de identificadores repetidos em determinadas características.

#### Análise de qualidade

Foram identificados 14 registros sem código de parâmetro.

A característica `Nitrate` apresentou correspondência entre quantidade de registros e códigos distintos, sem identificadores ausentes.

Já `Turbidity`, `pH` e `Dissolved oxygen (DO)` apresentaram situações que merecem investigação adicional.

#### Conclusão

O perfilamento permitiu avaliar a cardinalidade, a distribuição e a completude dos parâmetros selecionados.

Os resultados demonstram que a quantidade de registros não representa necessariamente a quantidade de parâmetros únicos.

A identificação de códigos ausentes e repetidos fornece subsídios para as próximas etapas de validação e preparação dos dados.

Os identificadores ausentes deverão ser investigados antes de qualquer decisão de exclusão ou substituição.

## 13. Exibição completa das características encontradas

### Objetivo

Apresentar integralmente o resumo estatístico das características identificadas na pesquisa refinada de parâmetros da API USGS.

### Justificativa

Na etapa anterior, foi construído o DataFrame `resumo_parametros`, contendo a distribuição dos registros por grupo e nome da característica.

A visualização convencional de DataFrames no Jupyter Notebook pode limitar a quantidade de linhas apresentadas, dependendo das configurações de exibição.

Para garantir que todas as características identificadas sejam apresentadas, será utilizada a conversão do DataFrame para uma representação textual.

### Estratégia

1. Recuperar o DataFrame `resumo_parametros`, criado na Célula 12.
2. Converter seu conteúdo para uma representação textual utilizando `to_string()`.
3. Desabilitar a exibição dos índices com `index=False`.
4. Apresentar integralmente os resultados no notebook.

### Conceitos técnicos

**to_string():** converte um DataFrame em uma representação textual, permitindo controlar a apresentação dos dados.

**index=False:** impede que os índices do DataFrame sejam apresentados na saída.

**print():** exibe o conteúdo textual no notebook.

### Resultado esperado

Exibição completa das 24 características identificadas, incluindo grupo, quantidade de registros, códigos distintos e códigos ausentes.

In [13]:
# Exibir todas as características encontradas

print(
    resumo_parametros.to_string(index=False)
)

          characteristicGroup                           characteristicName  registros  codigos_distintos  codigos_ausentes
                     Nutrient                                   Phosphorus         40                 39                 1
                     Physical                                    Turbidity         25                 21                 2
                     Nutrient                                      Nitrate         21                 21                 0
                     Physical                                           pH         18                 15                 3
                     Physical                         Specific conductance         15                 13                 2
Inorganics, Major, Non-metals                        Dissolved oxygen (DO)          8                  6                 2
                     Nutrient     Inorganic nitrogen (nitrate and nitrite)          8                  7                 1
                

### 13.1. Resultado e conclusão da exploração

A apresentação completa do resumo estatístico foi executada com sucesso.

#### Resultado final

Foram apresentadas integralmente as 24 características identificadas durante a pesquisa refinada, totalizando 176 registros e 158 códigos distintos.

A tabela também apresenta a quantidade de códigos ausentes e a distribuição dos registros por grupo e característica.

#### Análise técnica

A visualização completa permite identificar variações de nomenclatura dentro de um mesmo conjunto de parâmetros.

Um exemplo é a característica `Temperature`, que apresenta diferentes classificações relacionadas ao meio ou às condições de medição.

Essa informação será importante para futuras decisões de padronização e modelagem dos dados.

#### Conclusão

A etapa de exploração da API USGS foi concluída, permitindo identificar o endpoint de características, inspecionar a estrutura JSON, transformar os registros em DataFrame e investigar os parâmetros relacionados à qualidade da água.

O resultado constitui uma base inicial para as próximas etapas do projeto Open Water Analytics, especialmente a definição dos critérios de seleção, validação e transformação dos dados.

A exploração não representa ainda a validação definitiva dos parâmetros ambientais, sendo necessária uma investigação complementar sobre os códigos e suas respectivas definições.

## 14. Descoberta das estações de monitoramento USGS

### Objetivo

Identificar estações de monitoramento disponibilizadas pela API USGS, investigando seus identificadores, informações geográficas e coordenadas.

### Justificativa

Após explorar o catálogo de características da USGS, precisamos identificar as estações responsáveis pelo monitoramento ambiental.

As coordenadas geográficas serão importantes para estabelecer uma relação espacial entre as estações de monitoramento e os dados meteorológicos disponibilizados pela Open-Meteo.

### Estratégia

1. Consultar o endpoint de estações de monitoramento da USGS.
2. Configurar a quantidade de registros retornados.
3. Executar a requisição HTTP utilizando a biblioteca Requests.
4. Validar a resposta recebida.
5. Converter o JSON em um objeto Python.
6. Transformar os registros GeoJSON em um DataFrame utilizando Pandas.
7. Inspecionar os campos e as informações retornadas.

### Conceitos técnicos

**GeoJSON:** formato utilizado para representar informações geográficas estruturadas em JSON.

**requests.get():** realiza uma requisição HTTP GET.

**raise_for_status():** verifica se a resposta HTTP representa uma condição de erro.

**json():** converte a resposta JSON em estruturas Python.

**pd.json_normalize():** transforma estruturas JSON em formato tabular, facilitando a análise.

### Resultado esperado

Identificar as estações disponíveis e inspecionar suas informações geográficas, especialmente latitude e longitude, para utilização nas próximas etapas do projeto.

In [16]:
# Endpoint de estações de monitoramento USGS
BASE_URL = "https://api.waterdata.usgs.gov/ogcapi/v1/collections/monitoring-locations/items"

# Parâmetros da consulta
PARAMS = {
    "limit": 5
}

# Requisição HTTP
response_locais = requests.get(
    BASE_URL,
    params=PARAMS,
    timeout=30
)

# Validação da resposta
response_locais.raise_for_status()

# Conversão para objeto Python
dados_locais = response_locais.json()

# Inspeção da resposta
print("Status HTTP:", response_locais.status_code)
print("Tipo de retorno:", dados_locais.get("type"))
print("Registros retornados:", dados_locais.get("numberReturned"))
print("Total de registros encontrados:", dados_locais.get("numberMatched"))

# Conversão para DataFrame
df_locais = pd.json_normalize(dados_locais["features"])

# Inspeção dos campos disponíveis
print("\nColunas disponíveis:")
print(df_locais.columns.tolist())

# Visualização dos registros
display(df_locais.head())

Status HTTP: 200
Tipo de retorno: FeatureCollection
Registros retornados: 5
Total de registros encontrados: None

Colunas disponíveis:
['type', 'id', 'geometry', 'properties.id', 'properties.agency_code', 'properties.agency_name', 'properties.monitoring_location_number', 'properties.monitoring_location_name', 'properties.district_code', 'properties.country_code', 'properties.country_name', 'properties.state_code', 'properties.state_name', 'properties.county_code', 'properties.county_name', 'properties.minor_civil_division_code', 'properties.site_type_code', 'properties.site_type', 'properties.hydrologic_unit_code', 'properties.basin_code', 'properties.altitude', 'properties.altitude_accuracy', 'properties.altitude_method_code', 'properties.altitude_method_name', 'properties.vertical_datum', 'properties.vertical_datum_name', 'properties.horizontal_positional_accuracy_code', 'properties.horizontal_positional_accuracy', 'properties.horizontal_position_method_code', 'properties.horizontal_

,type,id,geometry,properties.id,properties.agency_code,properties.agency_name,properties.monitoring_location_number,properties.monitoring_location_name,properties.district_code,properties.country_code,...,properties.construction_date,properties.aquifer_code,properties.national_aquifer_code,properties.aquifer_type_code,properties.well_constructed_depth,properties.hole_constructed_depth,properties.depth_source_code,properties.revision_note,properties.revision_created,properties.revision_modified
0,Feature,AL012-90100100001,None,AL012-90100100001,AL012,Alabama Office of Water Resources,90100100001,Autauga County Water Authority,01,US,...,None,None,None,None,None,None,None,None,None,None
1,Feature,AL012-90100100002,None,AL012-90100100002,AL012,Alabama Office of Water Resources,90100100002,Autaugaville Water System,01,US,...,None,None,None,None,None,None,None,None,None,None
2,Feature,AL012-90100100003,None,AL012-90100100003,AL012,Alabama Office of Water Resources,90100100003,Billingsley Water System,01,US,...,None,None,None,None,None,None,None,None,None,None
3,Feature,AL012-90100100005,None,AL012-90100100005,AL012,Alabama Office of Water Resources,90100100005,Water Works Board Of Prattville,01,US,...,None,None,None,None,None,None,None,None,None,None
4,Feature,AL012-90100100006,None,AL012-90100100006,AL012,Alabama Office of Water Resources,90100100006,West Autauga Water Authority,01,US,...,None,None,None,None,None,None,None,None,None,None



### Conclusão — Descoberta das estações de monitoramento USGS

A consulta ao endpoint de estações de monitoramento da API USGS foi executada com sucesso, retornando cinco registros no formato GeoJSON (`FeatureCollection`).

A conversão dos registros para um DataFrame utilizando `pd.json_normalize()` permitiu identificar 46 colunas, incluindo identificadores, informações das agências responsáveis e nomes das localizações monitoradas.

Durante a análise inicial, foram observados os seguintes pontos:

- Os cinco registros retornados pertencem a sistemas de abastecimento de água do estado do Alabama.
- A agência identificada foi a Alabama Office of Water Resources.
- O campo `geometry` apresentou valores ausentes nos registros analisados.
- O atributo `numberMatched` não foi disponibilizado na resposta recebida.

**Considerações técnicas**

A consulta demonstrou como consumir uma API com retorno GeoJSON e normalizar estruturas JSON em tabelas utilizando Pandas.

Entretanto, os primeiros registros retornados não representam necessariamente as estações adequadas ao objetivo da análise. Será necessário refinar os critérios de consulta para identificar localizações de monitoramento relevantes e verificar a disponibilidade de suas coordenadas geográficas.

**Resultado da etapa**

Foi realizada a primeira exploração do endpoint de localizações de monitoramento da USGS, identificando a estrutura dos dados e a necessidade de aplicar filtros específicos nas próximas consultas.
